# nb_fabric_log_diagnostics — turn "this cell is slow" into a ranked diagnosis
**What this is:** a client for the Fabric Spark Monitoring REST API (mirrors the OSS Spark History
Server contract, plus Fabric-only Advisor/resourceUsage/logs endpoints — Sec 37 of the internals doc)
that automates the seven-step manual workflow: list jobs → find the slow one → its stages → skew
check → spill check → Spark Advisor findings → resource usage → **one ranked diagnosis**.

**Honesty about what's tested:** the API client and every parsing/ranking function below are exercised
end to end against **schema-accurate simulated responses** (`DEMO_MODE`, on by default) — this proves
the logic. The live-Fabric HTTP calls themselves are **not** exercised here, because that requires a
running Fabric session and a real application id that this environment doesn't have. The request
shapes and endpoints are transcribed from the current Fabric documentation (Sec 37); verify against
your tenant's API version before relying on them for alerting.

In [1]:
NOTEBOOK_NAME = "nb_fabric_log_diagnostics"
DEMO_MODE     = True          # False = call the real Fabric REST API (needs a live session/app id)
WORKSPACE_ID  = "00000000-0000-0000-0000-000000000000"
ITEM_KIND     = "notebooks"   # notebooks | sparkJobDefinitions | lakehouses
ITEM_ID       = "11111111-1111-1111-1111-111111111111"
LIVY_ID       = "22222222-2222-2222-2222-222222222222"
APP_ID        = "application_1111111111110_0001"
ATTEMPT_ID    = 1
SPILL_MB_FLAG = 256          # stage-level spill above this is flagged
SKEW_RATIO    = 3.0          # max/median task duration above this is flagged as skew

In [2]:
import json, statistics, random
from datetime import datetime, timezone, timedelta

BASE = (f"v1/workspaces/{WORKSPACE_ID}/{ITEM_KIND}/{ITEM_ID}/livySessions/{LIVY_ID}"
        f"/applications/{APP_ID}/{ATTEMPT_ID}")
print(f"{NOTEBOOK_NAME} | demo_mode={DEMO_MODE}")
print("Base path:", BASE)

nb_fabric_log_diagnostics | demo_mode=True
Base path: v1/workspaces/00000000-0000-0000-0000-000000000000/notebooks/11111111-1111-1111-1111-111111111111/livySessions/22222222-2222-2222-2222-222222222222/applications/application_1111111111110_0001/1


## 1 — The REST client
One function, one contract: build the URL, call it, return parsed JSON. In Fabric, `client.get()`
comes from a pre-authenticated Fabric REST client available inside notebooks; outside Fabric (or for
scheduled jobs), the same call goes through an Entra app/SPN token exactly like the SQL Database
connectivity in `nb_metadata_sqldb_prototype`.

In [3]:
def fabric_get(path: str, params: dict = None):
    """GET against the Fabric Spark Monitoring API. Swap DEMO_MODE off in a live Fabric session."""
    if DEMO_MODE:
        return _demo_response(path, params or {})
    # --- live path (untested here - requires a running Fabric session) ---
    try:
        from sempy.fabric import FabricRestClient   # pre-authenticated inside a Fabric notebook
        client = FabricRestClient()
    except ImportError:
        import requests, os
        token = os.environ["FABRIC_TOKEN"]           # Entra app/SPN token - see Sec 27 pattern
        class _Client:
            def get(self, p, params=None):
                import requests as r
                return r.get(f"https://api.fabric.microsoft.com/{p}",
                             headers={"Authorization": f"Bearer {token}"}, params=params)
        client = _Client()
    resp = client.get(path, params=params)
    resp.raise_for_status() if hasattr(resp, "raise_for_status") else None
    return resp.json()
print("fabric_get ready")

fabric_get ready


## 2 — Demo backend: schema-accurate simulated responses

In [4]:
random.seed(11)

def _demo_jobs():
    now = datetime.now(timezone.utc)
    jobs = []
    # 8 fast jobs (cells 1-8) + one deliberately slow, skewed job (the "cell 9 is slow" scenario)
    for i in range(8):
        jobs.append({"jobId": i, "jobGroup": f"cell-{i+1}", "status": "SUCCEEDED",
                     "submissionTime": (now - timedelta(minutes=20-i)).isoformat(),
                     "completionTime": (now - timedelta(minutes=20-i-0.3)).isoformat(),
                     "numTasks": 40, "numCompletedTasks": 40, "numFailedTasks": 0,
                     "stageIds": [i]})
    jobs.append({"jobId": 8, "jobGroup": "cell-9", "status": "SUCCEEDED",
                "submissionTime": (now - timedelta(minutes=11)).isoformat(),
                "completionTime": now.isoformat(),
                "numTasks": 34, "numCompletedTasks": 34, "numFailedTasks": 0,
                "stageIds": [8, 9]})
    return jobs

def _demo_stages(job_id):
    if job_id != 8:
        return [{"stageId": job_id, "status": "COMPLETE", "numTasks": 40,
                 "shuffleReadBytes": 5_000_000, "shuffleWriteBytes": 5_000_000,
                 "memoryBytesSpilled": 0, "diskBytesSpilled": 0}]
    # the slow job: stage 9 is a wide shuffle with real skew and real spill
    return [{"stageId": 8, "status": "COMPLETE", "numTasks": 16,
             "shuffleReadBytes": 200_000_000, "shuffleWriteBytes": 900_000_000,
             "memoryBytesSpilled": 0, "diskBytesSpilled": 0},
            {"stageId": 9, "status": "COMPLETE", "numTasks": 18,
             "shuffleReadBytes": 900_000_000, "shuffleWriteBytes": 40_000_000,
             "memoryBytesSpilled": 640_000_000, "diskBytesSpilled": 310_000_000}]

def _demo_tasks(stage_id):
    if stage_id != 9:
        return [{"taskId": i, "duration": 800 + random.randint(-100, 100)} for i in range(16)]
    # one hot key: task 3 takes ~9x the median - the skew signature
    durs = [1200 + random.randint(-150, 150) for _ in range(17)]
    durs.append(11400)
    return [{"taskId": i, "duration": d} for i, d in enumerate(durs)]

def _demo_advice(job_id, stage_id):
    if job_id == 8 and stage_id == 9:
        return [{"id": 1, "level": "Warning", "name": "Data skew detected",
                 "description": "One or more tasks in this stage processed significantly more data "
                                "than others. Consider salting the join/aggregation key.",
                 "source": "system"},
                {"id": 2, "level": "Warning", "name": "Spill to disk detected",
                 "description": "Stage 9 spilled ~310 MB to disk. Reduce partition size or executor "
                                "core count to increase per-task memory.", "source": "system"}]
    return []

def _demo_resource_usage(job_group):
    if job_group == "cell-9":
        return {"jobGroup": job_group, "peakMemoryUtilPct": 96, "avgCpuUtilPct": 71,
               "executors": 4, "note": "memory near saturation during this cell"}
    return {"jobGroup": job_group, "peakMemoryUtilPct": 45, "avgCpuUtilPct": 38, "executors": 4}

def _demo_response(path, params):
    if path.endswith("/jobs"): return _demo_jobs()
    if "taskList" in path:
        # .../stages/{stageId}/{attemptId}/taskList
        stage_id = int(path.split("/stages/")[1].split("/")[0])
        return _demo_tasks(stage_id)
    if path.endswith("/stages"):
        job_id = params.get("jobId")
        return _demo_stages(int(job_id)) if job_id is not None else sum((_demo_stages(j) for j in range(9)), [])
    if path.endswith("/advice"):
        return _demo_advice(int(params.get("jobId", -1)), int(params.get("stageId", -1)))
    if path.endswith("/resourceUsage"):
        return _demo_resource_usage(params.get("jobGroup", ""))
    return {}
print("demo backend ready - simulates a 9-job application where cell-9 is the slow one")

demo backend ready - simulates a 9-job application where cell-9 is the slow one


## 3 — The workflow: jobs → slowest → stages → skew/spill → advice → resources → diagnosis
Each step is a thin wrapper over `fabric_get`. The ranking at the end is what turns five API calls
into one answer.

In [5]:
def duration_secs(job):
    a = datetime.fromisoformat(job["submissionTime"]); b = datetime.fromisoformat(job["completionTime"])
    return (b - a).total_seconds()

def list_jobs():
    return fabric_get(f"{BASE}/jobs")

def slowest_job(jobs):
    return max(jobs, key=duration_secs)

def stages_for_job(job_id):
    return fabric_get(f"{BASE}/stages", params={"jobId": job_id})

def tasks_for_stage(stage_id):
    return fabric_get(f"{BASE}/stages/{stage_id}/{ATTEMPT_ID}/taskList")

def advice_for(job_id, stage_id):
    return fabric_get(f"{BASE}/advice", params={"jobId": job_id, "stageId": stage_id})

def resource_usage(job_group):
    return fabric_get(f"{BASE}/resourceUsage", params={"jobGroup": job_group})

jobs = list_jobs()
slow = slowest_job(jobs)
print(f"slowest job: jobId={slow['jobId']} jobGroup={slow['jobGroup']!r} "
      f"duration={duration_secs(slow):.0f}s  <- this identifies the CELL")
assert slow["jobGroup"] == "cell-9", "expected the deliberately-slow demo job to surface"

slowest job: jobId=8 jobGroup='cell-9' duration=660s  <- this identifies the CELL


In [6]:
stages = sorted(stages_for_job(slow["jobId"]), key=lambda s: s["shuffleReadBytes"], reverse=True)
findings = []
for s in stages:
    tasks = tasks_for_stage(s["stageId"])
    durs = [t["duration"] for t in tasks]
    med = statistics.median(durs); mx = max(durs)
    ratio = mx / med if med else 1.0
    spill_mb = (s["memoryBytesSpilled"] + s["diskBytesSpilled"]) / 1024 / 1024
    if ratio >= SKEW_RATIO:
        findings.append(("SKEW", s["stageId"],
            f"max task {mx}ms vs median {med:.0f}ms (ratio {ratio:.1f}x) - one task dominates the stage"))
    if spill_mb >= SPILL_MB_FLAG:
        findings.append(("SPILL", s["stageId"], f"{spill_mb:.0f} MB spilled - partitions too large for available memory"))
    print(f"stage {s['stageId']}: tasks={len(tasks)} max/median={ratio:.1f}x spill={spill_mb:.0f}MB "
          f"shuffle_read={s['shuffleReadBytes']/1e6:.0f}MB")

advice = []
for s in stages:
    advice += advice_for(slow["jobId"], s["stageId"])
res = resource_usage(slow["jobGroup"])

print(f"\nfindings: {len(findings)} | advisor items: {len(advice)} | "
      f"peak memory util during this cell: {res.get('peakMemoryUtilPct')}%")
assert any(f[0] == "SKEW" for f in findings), "expected the skew signature in the demo stage"
assert any(f[0] == "SPILL" for f in findings), "expected the spill signature in the demo stage"

stage 9: tasks=18 max/median=8.9x spill=906MB shuffle_read=900MB
stage 8: tasks=16 max/median=1.1x spill=0MB shuffle_read=200MB

findings: 2 | advisor items: 2 | peak memory util during this cell: 96%


## 4 — The one thing you actually wanted: a ranked diagnosis

In [7]:
def diagnose(job, findings, advice, resource_usage):
    lines = [f"DIAGNOSIS for {job['jobGroup']} (jobId={job['jobId']}, {duration_secs(job):.0f}s)", "=" * 60]
    if not findings and not advice:
        lines.append("No skew, spill or advisor findings - this cell is likely slow because of DATA VOLUME,")
        lines.append("not an anti-pattern. Check shuffle bytes vs. pool size before adding nodes.")
        return "\n".join(lines)
    ranked = sorted(findings, key=lambda f: {"SKEW": 2, "SPILL": 1}.get(f[0], 0), reverse=True)
    for kind, stage_id, detail in ranked:
        fix = {"SKEW": "-> salt the key, or confirm AQE skew-join is on (spark.sql.adaptive.skewJoin.enabled)",
               "SPILL": "-> lower advisoryPartitionSizeInBytes, or fewer cores per executor for more memory/task"
               }[kind]
        lines.append(f"[{kind}] stage {stage_id}: {detail}\n       {fix}")
    for a in advice:
        lines.append(f"[ADVISOR:{a['level'].upper()}] {a['name']}: {a['description']}")
    if resource_usage.get("peakMemoryUtilPct", 0) >= 90:
        lines.append(f"[RESOURCE] peak memory {resource_usage['peakMemoryUtilPct']}% during this cell "
                     f"- confirms the spill finding rather than a separate cause.")
    lines.append("")
    lines.append(f"-> Start with the top-ranked finding; re-run this notebook after each fix to confirm "
                f"the signature is gone rather than guessing.")
    return "\n".join(lines)

print(diagnose(slow, findings, advice, res))

DIAGNOSIS for cell-9 (jobId=8, 660s)
[SKEW] stage 9: max task 11400ms vs median 1281ms (ratio 8.9x) - one task dominates the stage
       -> salt the key, or confirm AQE skew-join is on (spark.sql.adaptive.skewJoin.enabled)
[SPILL] stage 9: 906 MB spilled - partitions too large for available memory
       -> lower advisoryPartitionSizeInBytes, or fewer cores per executor for more memory/task
[ADVISOR:WARNING] Data skew detected: One or more tasks in this stage processed significantly more data than others. Consider salting the join/aggregation key.
[ADVISOR:WARNING] Spill to disk detected: Stage 9 spilled ~310 MB to disk. Reduce partition size or executor core count to increase per-task memory.
[RESOURCE] peak memory 96% during this cell - confirms the spill finding rather than a separate cause.

-> Start with the top-ranked finding; re-run this notebook after each fix to confirm the signature is gone rather than guessing.


## 5 — Switching to a live Fabric session
1. Set `DEMO_MODE = False`.
2. Set `WORKSPACE_ID` / `ITEM_ID` / `LIVY_ID` / `APP_ID` from the notebook's own session — in Fabric,
   `notebookutils` and the Monitoring hub's Recent runs page expose these (Livy Id and Application Id
   are shown directly in Recent runs).
3. Inside a Fabric notebook, `fabric_get` will use `sempy.fabric.FabricRestClient()` automatically —
   no manual token handling needed.
4. For a **scheduled** diagnostic (not run from inside the target notebook), authenticate with an
   Entra app/SPN token as in `nb_metadata_sqldb_prototype`'s SQL Database connection, and set
   `FABRIC_TOKEN` in the environment.
5. Everything from Section 3 onward runs unchanged — the workflow doesn't know or care whether the
   data came from the demo backend or the real API.

**Where to point this automatically:** wire it into `etl_run_log` (Sec 27/33) — when a run's
`duration_seconds` exceeds a threshold or its status is `FAILED`, call `diagnose()` on that run's
`spark_app_id` and attach the result to the run log entry. A failing pipeline then arrives with its
own root-cause analysis already attached, rather than sending someone to five tabs.

In [8]:
print("Demo run complete. Live-mode checklist:")
print(" [ ] DEMO_MODE = False")
print(" [ ] WORKSPACE_ID / ITEM_ID / LIVY_ID / APP_ID set from a real Recent runs entry")
print(" [ ] sempy available (Fabric notebooks have it by default) OR FABRIC_TOKEN set")
print(" [ ] verify endpoint paths against your tenant's current API version before alerting on them")

Demo run complete. Live-mode checklist:
 [ ] DEMO_MODE = False
 [ ] WORKSPACE_ID / ITEM_ID / LIVY_ID / APP_ID set from a real Recent runs entry
 [ ] sempy available (Fabric notebooks have it by default) OR FABRIC_TOKEN set
 [ ] verify endpoint paths against your tenant's current API version before alerting on them
